## 🎯 Learning Objectives
* Understand the fundamental components of a retrieval chain in LangChain.
* Implement a simple retrieval chain using LangChain Expression Language (LCEL).
* Integrate an in-memory vector store, a retriever, a prompt template, and an LLM to answer questions based on provided documents.


## Exercise: Building a Simple Retrieval Chain with LCEL

### Task Description

In this exercise, you will build your first retrieval chain using LangChain Expression Language (LCEL). A retrieval chain is a powerful pattern where an LLM first retrieves relevant information from a knowledge base (e.g., documents, databases) and then uses that information to generate a more informed and accurate answer. This is a core concept in building RAG (Retrieval Augmented Generation) applications.

Your goal is to construct a simple retrieval chain that can answer questions based on a small, predefined set of in-memory documents. You will use an in-memory vector store for simplicity.

### Requirements

1.  **Document Preparation**: Define a list of `Document` objects containing some sample text.
2.  **Embedding Model**: Initialize an embedding model (e.g., `OpenAIEmbeddings` or a local alternative).
3.  **Vector Store**: Create an in-memory vector store (e.g., `Chroma`) from your documents using the embedding model.
4.  **Retriever**: Convert the vector store into a retriever.
5.  **Prompt Template**: Define a `ChatPromptTemplate` that instructs the LLM to answer questions based on the provided context.
6.  **LLM**: Initialize a large language model (e.g., `ChatOpenAI` or a local LLM).
7.  **LCEL Chain**: Construct the retrieval chain using LCEL's `|` operator and `RunnablePassthrough`.
    *   The chain should accept a dictionary with a `question` key.
    *   It should first retrieve relevant documents based on the `question`.
    *   Then, it should pass both the original `question` and the retrieved `context` to the LLM via the prompt template.
    *   Finally, it should parse the LLM's output into a string.

### Evaluation Criteria

*   **Correctness**: The chain correctly integrates all specified components (documents, embeddings, vector store, retriever, prompt, LLM) using LCEL.
*   **Functionality**: The chain successfully processes a test question and returns a coherent answer based on the provided documents.
*   **LCEL Usage**: Effective and idiomatic use of LCEL for chaining components.
*   **Code Clarity**: The code is well-structured, readable, and includes comments where necessary.


In [ ]:
# Ensure you have the necessary packages installed:
# pip install langchain langchain-openai langchain-chroma

import os
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --- Configuration and Setup ---

# Set your OpenAI API key. In a real application, use environment variables or a secrets manager.
# For this exercise, you can uncomment and set it directly, or ensure it's set in your environment.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# Check if the API key is set
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY environment variable not set. Please set it to run this exercise.")

# 1. Define sample documents
# These documents describe a fictional company, 'AgenticLabs.ng', and its offerings.
raw_documents = [
    Document(page_content="AgenticLabs.ng is a leading AI research and development company focused on building advanced agentic AI systems. We specialize in creating autonomous agents that can perform complex tasks."),
    Document(page_content="Our core products include the 'Cognito Agent Platform' for enterprise automation and 'Synapse AI' for intelligent data analysis. We also offer consulting services for AI strategy."),
    Document(page_content="The Cognito Agent Platform leverages LangChain for orchestrating LLM-powered workflows, enabling businesses to automate customer support, data processing, and more."),
    Document(page_content="Synapse AI is designed for large-scale data ingestion and real-time analytics, providing actionable insights for decision-makers. It integrates with various data sources."),
    Document(page_content="AgenticLabs.ng was founded in 2023 by a team of AI pioneers. Our mission is to democratize access to advanced AI capabilities.")
]

# 2. Initialize an embedding model
# OpenAIEmbeddings is a popular choice. For local development, you might use HuggingFaceEmbeddings.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small") # Using a modern, efficient embedding model

# 3. Create an in-memory vector store
# Chroma is a lightweight, in-memory vector store suitable for exercises.
vectorstore = Chroma.from_documents(documents=raw_documents, embedding=embeddings)

# 4. Create a retriever from the vector store
# The 'as_retriever()' method converts the vector store into a retriever component.
retriever = vectorstore.as_retriever()

# 5. Define the ChatPromptTemplate
# This template guides the LLM on how to use the retrieved context.
# The 'context' variable will be populated by the retriever, and 'question' by the user input.
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an AI assistant for AgenticLabs.ng. Answer the user's question based *only* on the provided context. If the answer is not in the context, state that you don't have enough information."),
    ("human", "Context: {context}\nQuestion: {question}")
])

# 6. Initialize the Large Language Model (LLM)
# We'll use ChatOpenAI, a powerful conversational model.
llm = ChatOpenAI(model="gpt-4o", temperature=0.1) # Using a modern, capable LLM

print("Setup complete. You can now proceed to implement the retrieval chain.")


## Your Turn! Implement the Retrieval Chain

Now it's your turn to combine the `retriever`, `prompt`, `llm`, and `StrOutputParser` using LangChain Expression Language (LCEL).

Your chain should:
1.  Accept an input dictionary with a `question` key.
2.  Use `RunnablePassthrough` to pass the `question` directly to the prompt.
3.  Use the `retriever` to fetch relevant `context` based on the `question`.
4.  Format the retrieved documents into a single string for the `context` variable in the prompt.
5.  Pass the `context` and `question` to the `prompt`.
6.  Send the templated prompt to the `llm`.
7.  Parse the `llm`'s output into a string.

Think about how to structure the input to the prompt (a dictionary with `context` and `question` keys) and how to use `RunnablePassthrough` to manage the flow of information.


In [ ]:
# --- Reference Solution: Building the Retrieval Chain ---

# Helper function to format retrieved documents into a single string.
# This is crucial because the prompt expects a single 'context' string, not a list of Document objects.
def format_docs(docs):
    # Join the page_content of each document with two newlines for readability.
    return "\n\n".join(doc.page_content for doc in docs)

# Construct the RAG chain using LCEL
# The chain is defined from right to left, or rather, as a sequence of operations.
rag_chain = (
    # Step 1: Prepare the input for the prompt.
    # We need a dictionary with 'context' and 'question' keys.
    # 'question' comes directly from the initial input to the chain.
    # 'context' comes from the retriever, which itself uses the initial 'question'.
    {
        "context": retriever | format_docs, # First, retrieve documents, then format them.
        "question": RunnablePassthrough() # Pass the original 'question' through directly.
    }
    # Step 2: Pass the prepared input to the prompt template.
    | prompt
    # Step 3: Send the templated prompt (now a ChatPromptValue) to the LLM.
    | llm
    # Step 4: Parse the LLM's ChatMessage output into a simple string.
    | StrOutputParser()
)

# --- Test the Retrieval Chain ---

print("\n--- Testing the Retrieval Chain ---\n")

# Test Question 1: Directly answerable from documents
question1 = "What is AgenticLabs.ng's mission?"
print(f"Question 1: {question1}")
response1 = rag_chain.invoke({"question": question1})
print(f"Answer 1: {response1}\n")

# Test Question 2: Requires combining information
question2 = "What are the main products offered by AgenticLabs.ng and what do they do?"
print(f"Question 2: {question2}")
response2 = rag_chain.invoke({"question": question2})
print(f"Answer 2: {response2}\n")

# Test Question 3: Not directly answerable from documents (should state lack of info)
question3 = "Who is the CEO of AgenticLabs.ng?"
print(f"Question 3: {question3}")
response3 = rag_chain.invoke({"question": question3})
print(f"Answer 3: {response3}\n")

# Test Question 4: About a specific product's technology
question4 = "Which technology does the Cognito Agent Platform leverage for workflows?"
print(f"Question 4: {question4}")
response4 = rag_chain.invoke({"question": question4})
print(f"Answer 4: {response4}\n")

print("Retrieval chain testing complete.")
